In [11]:
#https://github.com/wpilibsuite/allwpilib/tree/main/apriltag/src/main/native/resources/edu/wpi/first/apriltag
#https://tools.limelightvision.io/apriltag-generator

In [2]:
import requests

# The "Raw" URL for the file you found in that folder
url = "https://raw.githubusercontent.com/wpilibsuite/allwpilib/main/apriltag/src/main/native/resources/edu/wpi/first/apriltag/2025-reefscape-welded.json"
filename = "2025-reefscape-welded.json"

print(f"Downloading real field data from: {url}")

try:
    r = requests.get(url)
    if r.status_code == 200:
        with open(filename, 'wb') as f:
            f.write(r.content)
        print("✅ Success! Real 2025 Field Layout downloaded.")
        print("You can now re-run your FieldMapper code.")
    else:
        print(f"❌ Error: GitHub returned status {r.status_code}")
except Exception as e:
    print(f"❌ Failed: {e}")

✅ Success! Real 2025 Field Layout downloaded.
You can now re-run your FieldMapper code.


In [1]:
import json
import numpy as np
import cv2
import math
import os
import requests
from pupil_apriltags import Detector

JSON_FILE = "2025-reefscape-welded.json"

# ================= CLASS: FIELD MAPPER =================
class FieldMapper:
    def __init__(self, json_path):
        self.tag_poses = {} # Stores 4x4 Matrices
        self.load_json(json_path)

    def load_json(self, path):
        with open(path, 'r') as f:
            data = json.load(f)
        
        for tag in data['tags']:
            id = tag['ID']
            # Extract Translation
            tx = tag['pose']['translation']['x']
            ty = tag['pose']['translation']['y']
            tz = tag['pose']['translation']['z']
            
            # Extract Quaternion
            qw = tag['pose']['rotation']['quaternion']['W']
            qx = tag['pose']['rotation']['quaternion']['X']
            qy = tag['pose']['rotation']['quaternion']['Y']
            qz = tag['pose']['rotation']['quaternion']['Z']
            
            # Convert to 4x4 Matrix (Field -> Tag)
            self.tag_poses[id] = self.create_matrix(tx, ty, tz, qw, qx, qy, qz)
            
    def create_matrix(self, tx, ty, tz, qw, qx, qy, qz):
        # Rotation Matrix from Quaternion
        # Standard conversion formula
        xx, yy, zz = qx*qx, qy*qy, qz*qz
        xy, xz, yz = qx*qy, qx*qz, qy*qz
        wx, wy, wz = qw*qx, qw*qy, qw*qz

        R = np.array([
            [1 - 2*(yy+zz), 2*(xy-wz),   2*(xz+wy)],
            [2*(xy+wz),     1 - 2*(xx+zz), 2*(yz-wx)],
            [2*(xz-wy),     2*(yz+wx),     1 - 2*(xx+yy)]
        ])
        
        # Full 4x4 Transform
        T = np.eye(4)
        T[:3, :3] = R
        T[:3, 3] = [tx, ty, tz]
        return T

    def get_cam_position(self, tag_id, r_matrix, t_vec):
        """
        Calculates Camera Position in Field Coordinates.
        tag_id: The ID detected
        r_matrix: The 3x3 rotation from AprilTag library
        t_vec: The 3x1 translation from AprilTag library
        """
        if tag_id not in self.tag_poses:
            return None
        
        # 1. Get Transform: Camera -> Tag
        # Note: pupil_apriltags output is T_camera_tag
        T_cam_tag = np.eye(4)
        T_cam_tag[:3, :3] = r_matrix
        T_cam_tag[:3, 3] = t_vec.flatten()
        
        # 2. Invert it to get: Tag -> Camera
        T_tag_cam = np.linalg.inv(T_cam_tag)
        
        # 3. Get Transform: Field -> Tag
        T_field_tag = self.tag_poses[tag_id]
        
        # 4. Chain them: Field -> Tag -> Camera
        T_field_cam = np.matmul(T_field_tag, T_tag_cam)
        
        # 5. Extract Camera Position (x, y, z)
        cam_x = T_field_cam[0, 3]
        cam_y = T_field_cam[1, 3]
        cam_z = T_field_cam[2, 3]
        
        return (cam_x, cam_y, cam_z)

# ================= MAIN EXECUTION =================

# 1. Setup Field Map
mapper = FieldMapper(JSON_FILE)
print(f"Loaded Field Map with {len(mapper.tag_poses)} tags.")

# 2. Setup Camera
TAG_SIZE = 0.0508 # 2 inches
CAMERA_PARAMS = [600, 600, 320, 240]
cap = cv2.VideoCapture(0)
detector = Detector(families='tag36h11')

print("Starting Localization... (Press 'q' to quit)")

try:
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        tags = detector.detect(gray, estimate_tag_pose=True, camera_params=CAMERA_PARAMS, tag_size=TAG_SIZE)
        
        for tag in tags:
            # Draw
            corners = tag.corners.astype(int)
            for i in range(4):
                cv2.line(frame, tuple(corners[i]), tuple(corners[(i+1)%4]), (0, 255, 0), 2)
            
            # CALCULATE FIELD POSITION
            result = mapper.get_cam_position(tag.tag_id, tag.pose_R, tag.pose_t)
            
            if result:
                cx, cy, cz = result
                
                # Print to Stdout (Overwriting line)
                status = f"ID:{tag.tag_id} | ROBOT POS -> X: {cx:.2f}m, Y: {cy:.2f}m, Z: {cz:.2f}m"
                print(status)
                
                # Draw on Screen
                cv2.putText(frame, f"X:{cx:.2f} Y:{cy:.2f}", (corners[0][0], corners[0][1]-20), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        cv2.imshow('Localization', frame)
        if cv2.waitKey(1) == ord('q'):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

Loaded Field Map with 22 tags.
Starting Localization... (Press 'q' to quit)
ID:1 | ROBOT POS -> X: 16.64m, Y: 0.61m, Z: 1.35m
ID:1 | ROBOT POS -> X: 16.65m, Y: 0.62m, Z: 1.34m
ID:1 | ROBOT POS -> X: 16.65m, Y: 0.63m, Z: 1.34m
ID:1 | ROBOT POS -> X: 16.65m, Y: 0.63m, Z: 1.34m
ID:1 | ROBOT POS -> X: 16.64m, Y: 0.63m, Z: 1.34m
ID:1 | ROBOT POS -> X: 16.64m, Y: 0.63m, Z: 1.34m
ID:1 | ROBOT POS -> X: 16.64m, Y: 0.63m, Z: 1.34m
ID:1 | ROBOT POS -> X: 16.64m, Y: 0.64m, Z: 1.34m
ID:1 | ROBOT POS -> X: 16.64m, Y: 0.64m, Z: 1.34m
ID:1 | ROBOT POS -> X: 16.63m, Y: 0.64m, Z: 1.34m
ID:1 | ROBOT POS -> X: 16.63m, Y: 0.64m, Z: 1.35m
ID:1 | ROBOT POS -> X: 16.63m, Y: 0.63m, Z: 1.35m
ID:1 | ROBOT POS -> X: 16.63m, Y: 0.62m, Z: 1.35m
ID:1 | ROBOT POS -> X: 16.63m, Y: 0.62m, Z: 1.36m
ID:1 | ROBOT POS -> X: 16.62m, Y: 0.63m, Z: 1.36m
ID:1 | ROBOT POS -> X: 16.62m, Y: 0.63m, Z: 1.36m
ID:1 | ROBOT POS -> X: 16.62m, Y: 0.63m, Z: 1.36m
ID:1 | ROBOT POS -> X: 16.62m, Y: 0.63m, Z: 1.35m
ID:1 | ROBOT POS -> X: 1

KeyboardInterrupt: 